# jsonschema_specifications Package Testing

This notebook tests the jsonschema_specifications package, which provides JSON Schema specification documents and meta-schemas for validation.

## Example 1: Schema Specification Discovery

In [ ]:
# Example 1: Exploring jsonschema_specifications (safe approach)
import jsonschema_specifications

print('jsonschema_specifications imported successfully')
attrs = [attr for attr in dir(jsonschema_specifications) if not attr.startswith('_')]
print(f'Available attributes: {attrs}')

# Test basic functionality if available
if hasattr(jsonschema_specifications, 'REGISTRY'):
    print('\nRegistry available:')
    registry = jsonschema_specifications.REGISTRY
    print(f'Registry type: {type(registry)}')
    if hasattr(registry, 'keys'):
        print(f'Registry keys: {list(registry.keys())[:5]}...')
    elif hasattr(registry, '__iter__'):
        print(f'Registry items: {list(registry)[:5]}...')

print('\nBasic jsonschema_specifications exploration complete')


## Example 2: Meta-Schema Validation

In [ ]:
import jsonschema_specifications
import jsonschema
import json

# Test meta-schema validation
test_schemas = [
    # Valid schema
    {
        "$schema": "http://json-schema.org/draft-07/schema#",
        "type": "object",
        "properties": {
            "name": {"type": "string"},
            "age": {"type": "integer"}
        }
    },
    # Invalid schema (bad type)
    {
        "$schema": "http://json-schema.org/draft-07/schema#", 
        "type": "invalid_type",
        "properties": {
            "name": {"type": "string"}
        }
    },
    # Schema with references
    {
        "$schema": "http://json-schema.org/draft-07/schema#",
        "definitions": {
            "person": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"}
                }
            }
        },
        "type": "array",
        "items": {"$ref": "#/definitions/person"}
    }
]

print("Testing meta-schema validation:")

# Get the Draft 7 meta-schema
try:
    registry = jsonschema_specifications.REGISTRY
    draft7_url = "http://json-schema.org/draft-07/schema"

    assert draft7_url in registry, f"{draft7_url} missing from jsonschema_specifications.REGISTRY"
    draft7_spec = registry[draft7_url]
    draft7_meta = draft7_spec.contents
    meta_validator = jsonschema.Draft7Validator(draft7_meta)

    verdicts = []
    for i, schema in enumerate(test_schemas, 1):
        print(f"\nSchema {i}:")
        print(f"Schema: {json.dumps(schema, indent=2)[:200]}...")

        try:
            meta_validator.validate(schema)
            verdicts.append("valid")
            print("✅ Valid schema according to meta-schema")
        except jsonschema.ValidationError as e:
            verdicts.append(e.validator)
            print(f"❌ Meta-schema validation error: {e.message}")

    assert verdicts == ["valid", "anyOf", "valid"], verdicts
            
except Exception as e:
    print(f"Error setting up meta-schema validation: {e}")
    raise AssertionError(
        f"jsonschema_specifications example (cell 4) failed: {type(e).__name__}: {e}"
    ) from e

## Example 3: Specification Version Comparison

In [ ]:
import jsonschema_specifications
import json

# Compare different specification versions
def analyze_specification(spec_name, spec):
    print(f"\n=== {spec_name} ===")
    # `spec` is a referencing.Resource: `contents` is an attribute, while `id()` and `anchors()` are METHODS
    try:
        content = spec.contents
        print(f"Schema ID: {spec.id()}")
        assert spec.id().startswith("http://json-schema.org/draft-"), spec.id()
        assert isinstance(content, dict) and "$schema" in content, type(content)
        
        # Analyze schema structure
        if isinstance(content, dict):
            print(f"Top-level keys: {list(content.keys())}")
            
            # Check for properties
            if "properties" in content:
                props = content["properties"]
                print(f"Property count: {len(props)}")
                print(f"Sample properties: {list(props.keys())[:5]}")
            
            # Check for definitions
            if "definitions" in content:
                defs = content["definitions"]
                print(f"Definition count: {len(defs)}")
                print(f"Sample definitions: {list(defs.keys())[:5]}")
                
        anchors = list(spec.anchors())
        print(f"Anchors available: {len(anchors)}")
        
    except Exception as e:
        print(f"Error analyzing {spec_name}: {e}")
        raise

# Analyze available specifications
print("Analyzing JSON Schema specifications:")

# Define the specification URLs
spec_urls = {
    "DRAFT7": "http://json-schema.org/draft-07/schema",
    "DRAFT6": "http://json-schema.org/draft-06/schema", 
    "DRAFT4": "http://json-schema.org/draft-04/schema",
}

registry = jsonschema_specifications.REGISTRY

for spec_name, spec_url in spec_urls.items():
    try:
        if spec_url in registry:
            spec = registry[spec_url]
            analyze_specification(spec_name, spec)
        else:
            print(f"\n=== {spec_name} ===")
            print("❌ Not available in registry")
    except Exception as e:
        print(f"\n=== {spec_name} ===")
        print(f"❌ Error: {e}")
        raise  # any failure analysing a spec fails the cell

# Test specification features
print("\n" + "="*50)
print("Feature comparison:")
features = ["$id", "$schema", "definitions", "properties", "allOf", "anyOf", "oneOf"]

for feature in features:
    print(f"\n{feature}:")
    for spec_name, spec_url in spec_urls.items():
        try:
            if spec_url in registry:
                spec = registry[spec_url]
                if isinstance(spec.contents, dict):
                    has_feature = feature in str(spec.contents)
                    status = "✅" if has_feature else "❌"
                    print(f"  {spec_name}: {status}")
                else:
                    print(f"  {spec_name}: ❌")
            else:
                print(f"  {spec_name}: ❌")
        except:
            print(f"  {spec_name}: ❌")
            raise

## Example 4: Schema Anchor Resolution

In [ ]:
import jsonschema_specifications
import json

# Test schema anchor resolution
def explore_anchors(spec_name, spec):
    print(f"\n=== {spec_name} Anchors ===")
    try:
        anchors = spec.anchors
        print(f"Total anchors: {len(anchors)}")
        
        # Show sample anchors
        anchor_items = list(anchors.items())[:10]
        for anchor_name, anchor_value in anchor_items:
            print(f"  {anchor_name}: {type(anchor_value).__name__}")
            
        # Try to access specific common anchors
        common_anchors = ["meta", "schema", "properties", "items", "type"]
        print(f"\nCommon anchors:")
        for anchor in common_anchors:
            if anchor in anchors:
                print(f"  ✅ {anchor}")
            else:
                print(f"  ❌ {anchor}")
                
    except Exception as e:
        print(f"Error exploring anchors: {e}")

# Explore anchors in different specifications
specs_to_explore = ["DRAFT7", "DRAFT4"]
for spec_name in specs_to_explore:
    try:
        spec = getattr(jsonschema_specifications, spec_name, None)
        if spec:
            explore_anchors(spec_name, spec)
    except Exception as e:
        print(f"Error with {spec_name}: {e}")
        raise

# Test anchor-based validation
print("\n" + "="*50)
print("Testing anchor-based schema resolution:")

try:
    # Create a schema that uses anchors
    schema_with_anchor = {
        "$schema": "http://json-schema.org/draft-07/schema#",
        "$id": "http://example.com/schema",
        "type": "object",
        "properties": {
            "name": {"$ref": "#/definitions/name"},
            "items": {"$ref": "#/definitions/string_array"}
        },
        "definitions": {
            "name": {"type": "string", "minLength": 1},
            "string_array": {
                "type": "array",
                "items": {"type": "string"}
            }
        }
    }
    
    print(f"Schema with anchors: {json.dumps(schema_with_anchor, indent=2)[:300]}...")
    
    # Test validation with the schema
    test_data = {
        "name": "Test Item",
        "items": ["item1", "item2", "item3"]
    }
    
    import jsonschema
    jsonschema.validate(test_data, schema_with_anchor)
    print("\n✅ Anchor-based validation successful")
    
except Exception as e:
    print(f"\n❌ Anchor validation error: {e}")
    raise

## Example 5: Specification Introspection

In [ ]:
import jsonschema_specifications
import json
from collections import Counter

# Deep introspection of specification contents
def deep_analyze_spec(spec_name, spec):
    print(f"\n{'='*20} {spec_name} Analysis {'='*20}")
    
    # `spec` is a referencing.Resource: `contents` is an attribute, while `id()` and `anchors()` are METHODS
    try:
        content = spec.contents
        
        # Basic info
        print(f"ID: {spec.id()}")
        assert spec.id().startswith("http://json-schema.org/draft-"), spec.id()
        print(f"Content type: {type(content)}")
        
        if isinstance(content, dict):
            # Analyze structure depth
            def get_depth(obj, current_depth=0):
                if isinstance(obj, dict):
                    if not obj:
                        return current_depth
                    return max(get_depth(v, current_depth + 1) for v in obj.values())
                elif isinstance(obj, list):
                    if not obj:
                        return current_depth
                    return max(get_depth(item, current_depth + 1) for item in obj)
                else:
                    return current_depth
            
            depth = get_depth(content)
            print(f"Schema depth: {depth}")
            
            # Count different types of values
            def count_values(obj, counter=None):
                if counter is None:
                    counter = Counter()
                
                if isinstance(obj, dict):
                    for key, value in obj.items():
                        counter[f"key:{key}"] += 1
                        counter[f"type:{type(value).__name__}"] += 1
                        count_values(value, counter)
                elif isinstance(obj, list):
                    counter[f"type:list_length_{len(obj)}"] += 1
                    for item in obj:
                        count_values(item, counter)
                
                return counter
            
            value_counts = count_values(content)
            
            # Show most common keys
            common_keys = [(k, v) for k, v in value_counts.items() if k.startswith("key:")]
            common_keys.sort(key=lambda x: x[1], reverse=True)
            print(f"\nMost common keys:")
            for key, count in common_keys[:10]:
                print(f"  {key[4:]}: {count}")
            
            # Show value type distribution
            type_counts = [(k, v) for k, v in value_counts.items() if k.startswith("type:")]
            type_counts.sort(key=lambda x: x[1], reverse=True)
            print(f"\nValue type distribution:")
            for type_name, count in type_counts[:8]:
                print(f"  {type_name[5:]}: {count}")
                
        # Analyze anchors
        print(f"\nAnchor analysis:")
        anchors = list(spec.anchors())
        print(f"  Total anchors: {len(anchors)}")
        
        # Sample anchor types (each item is an Anchor object carrying .name and .resource)
        anchor_types = Counter()
        for anchor in anchors[:20]:
            anchor_types[type(anchor).__name__] += 1
        
        print(f"  Anchor value types: {dict(anchor_types)}")
        
    except Exception as e:
        print(f"Analysis error: {e}")
        raise

# Perform deep analysis on available specifications
available_specs = []
# Map spec names to their URLs
spec_urls = {
    "DRAFT7": "http://json-schema.org/draft-07/schema",
    "DRAFT6": "http://json-schema.org/draft-06/schema",
    "DRAFT4": "http://json-schema.org/draft-04/schema",
    "DRAFT3": "http://json-schema.org/draft-03/schema"
}

registry = jsonschema_specifications.REGISTRY

for spec_name, spec_url in spec_urls.items():
    try:
        if spec_url in registry:
            spec = registry[spec_url]
            available_specs.append((spec_name, spec))
    except Exception as e:
        print(f"Error loading {spec_name}: {e}")
        raise

print(f"Analyzing {len(available_specs)} specifications:")
for spec_name, spec in available_specs:
    deep_analyze_spec(spec_name, spec)

# Summary comparison
print(f"\n{'='*60}\"")
print("SPECIFICATION SUMMARY")
print(f"{'='*60}")
for spec_name, spec in available_specs:
    content_size = len(str(spec.contents))
    anchor_count = len(list(spec.anchors()))
    assert content_size > 0, spec_name
    print(f"{spec_name:>10}: {content_size:>8} chars, {anchor_count:>4} anchors")

print(f"Analyzing {len(available_specs)} specifications:")
for spec_name, spec in available_specs:
    deep_analyze_spec(spec_name, spec)